<h1 align="center" style="color:green">Fraud Detection Project Modeling and Evaluation</h1>

In [16]:
import pandas as pd

from insurance_fraud_detection.config import RAW_CSV, TARGET, DROP_COLUMNS, CLEANER_PARAMS_PATH
from insurance_fraud_detection.data_prep import load_data, split_data, drop_bad_rows
from insurance_fraud_detection.preprocessing import fit_cleaner, prepare
from insurance_fraud_detection.features import WOEEncoder

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
df = load_data()
train_df, test_df = split_data(df)
print(f"Train shape: {train_df.shape},\nTest shape: {test_df.shape}")

train_df = drop_bad_rows(train_df)
test_df = drop_bad_rows(test_df)
print(f"Train shape: {train_df.shape},\nTest shape: {test_df.shape}")

params = fit_cleaner(train_df)

Train shape: (12336, 31),
Test shape: (3084, 31)
Train shape: (12335, 31),
Test shape: (3084, 31)


In [12]:
train_ord = prepare(train_df, params)
test_ord = prepare(test_df, params)

print(f"train_ord shape: {train_ord.shape},\ntest_ord shape: {test_ord.shape}")  # expect (12335, 31) and (3084, 31)

train_ord shape: (12335, 31),
test_ord shape: (3084, 31)


In [14]:
from insurance_fraud_detection.preprocessing import load_params
assert load_params(CLEANER_PARAMS_PATH) == params
loaded = load_params(CLEANER_PARAMS_PATH)
print(loaded, params, loaded == params)

{'age': 39.0} {'age': 39.0} True


## Feature Engineering & Feature Selection
- WOE & IV


In [17]:
CANDIDATES = [c for c in train_ord.columns if c not in {TARGET, "policy_type", "age"}]

enc = WOEEncoder(cols=CANDIDATES).fit(train_ord[CANDIDATES], train_ord[TARGET])
iv = pd.Series(enc.iv_).sort_values(ascending=False)
print(iv.round(3))

base_policy              0.771
fault                    0.528
vehicle_category         0.491
make                     0.075
past_number_of_claims    0.064
address_change_claim     0.063
deductible               0.059
vehicle_price            0.055
month                    0.051
month_claimed            0.048
age_of_policy_holder     0.028
number_of_suppliments    0.028
agent_type               0.023
age_of_vehicle           0.023
sex                      0.017
year                     0.010
accident_area            0.010
day_of_week              0.009
days_policy_accident     0.009
days_policy_claim        0.006
day_of_week_claimed      0.006
police_report_filed      0.005
week_of_month            0.005
week_of_month_claimed    0.003
number_of_cars           0.001
witness_present          0.001
driver_rating            0.001
marital_status           0.001
dtype: float64


In [18]:
selected = iv[iv >= 0.02].index.tolist()   # 14 columns
selected.remove("vehicle_category")        # proxy for base_policy
print(len(selected), selected)

13 ['base_policy', 'fault', 'make', 'past_number_of_claims', 'address_change_claim', 'deductible', 'vehicle_price', 'month', 'month_claimed', 'age_of_policy_holder', 'number_of_suppliments', 'agent_type', 'age_of_vehicle']


- VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant

def vif_table(woe_df: pd.DataFrame) -> pd.Series:
    X = add_constant(woe_df)  # without the constant, VIF values are distorted
    vif = {
        col: variance_inflation_factor(X.values, i) # type: ignore
        for i, col in enumerate(X.columns) # type: ignore
        if col != "const"
    }
    return pd.Series(vif).sort_values(ascending=False)

In [21]:

from insurance_fraud_detection.config import SELECTED_COLS


woe_train = (
    WOEEncoder(cols=SELECTED_COLS)
    .fit(train_ord[SELECTED_COLS], train_ord[TARGET])
    .transform(train_ord[SELECTED_COLS])
)
print(vif_table(woe_train).round(1))

deductible               4.5
address_change_claim     4.5
month_claimed            2.6
month                    2.6
age_of_vehicle           1.3
age_of_policy_holder     1.3
base_policy              1.2
past_number_of_claims    1.2
vehicle_price            1.1
fault                    1.1
number_of_suppliments    1.0
agent_type               1.0
make                     1.0
dtype: float64
